# BiLSTM 

In [ ]:
from pathlib import Path

import re
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


device = torch.device("mps")

In [18]:
# гиперпараметры в одном месте
MAX_LEN    = 200       # обрезаем/дополняем каждый коммент до стольких токенов
MAX_VOCAB  = 50_000    # держим только N самых частых слов (+ pad, unk)
MIN_FREQ   = 2         # слова реже этого выкидываем
EMBED_DIM  = 100       # размерность GloVe
HIDDEN_DIM = 128       # размер скрытого состояния LSTM (на одно направление)
DROPOUT    = 0.3
BATCH_SIZE = 64
EPOCHS     = 4
LR         = 1e-3
VAL_SIZE   = 0.1

PAD, PAD_IDX = "<pad>", 0   # pad с индексом 0 — пустые позиции дадут нулевой эмбеддинг
UNK, UNK_IDX = "<unk>", 1   # unk ловит слова вне словаря

GLOVE_PATH = "../data/glove/glove.6B.100d.txt"

# Данные

In [ ]:
train = pd.read_csv("../data/train_clean.csv")   # уже почищены в ноутбуке 01
test = pd.read_csv("../data/test_clean.csv")
print("train:", train.shape, "| test:", test.shape)
train.head(3)

In [ ]:
# текст уже почищен в ноутбуке 01 (нижний регистр, без мусора) — здесь только токенизация
# пунктуацию .,!? выносим в отдельные токены (в GloVe для них есть векторы), а не клеим к словам
def tokenize(text):
    if not isinstance(text, str):   # NaN / не строка
        return []
    return re.findall(r"[a-z0-9']+|[.,!?]", text)

train["tokens"] = train["comment_text"].apply(tokenize)
test["tokens"] = test["comment_text"].apply(tokenize)
print(train["tokens"].iloc[0][:20])

In [5]:
# делим train/val ДО построения словаря — val честно играет роль новых данных
tr_idx, va_idx = train_test_split(np.arange(len(train)), test_size=VAL_SIZE,
                                  random_state=SEED, shuffle=True)
tr = train.iloc[tr_idx].reset_index(drop=True)
va = train.iloc[va_idx].reset_index(drop=True)
print("train:", len(tr), "| val:", len(va))

train: 143613 | val: 15958


# Словарь

In [6]:
# сеть не ест строки — нужен словарь слово -> id; частоты считаем только по train
counter = Counter()
for toks in tr["tokens"]:
    counter.update(toks)
print("уникальных слов:", len(counter))

itos = [PAD, UNK]                       # index -> слово, первыми спецтокены
for word, freq in counter.most_common():
    if freq < MIN_FREQ or len(itos) >= MAX_VOCAB:
        break                           # most_common отсортирован — можно прерваться
    itos.append(word)

stoi = {w: i for i, w in enumerate(itos)}   # слово -> index
VOCAB_SIZE = len(itos)
print("размер словаря:", VOCAB_SIZE)

уникальных слов: 187424
размер словаря: 50000


In [7]:
# кодируем коммент в фикс. длину: id + обрезка до MAX_LEN + дополнение pad справа
def encode(tokens, max_len=MAX_LEN):
    ids = [stoi.get(t, UNK_IDX) for t in tokens[:max_len]]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

# матрицы id считаем заранее (быстрее, чем кодировать внутри DataLoader)
X_tr = np.array([encode(t) for t in tr["tokens"]], dtype=np.int64)
X_va = np.array([encode(t) for t in va["tokens"]], dtype=np.int64)
X_te = np.array([encode(t) for t in test["tokens"]], dtype=np.int64)

# метки float32 (N, 6) — BCELoss сравнивает вероятность с целью как float
y_tr = tr[LABELS].values.astype(np.float32)
y_va = va[LABELS].values.astype(np.float32)
print("X_tr:", X_tr.shape, "| y_tr:", y_tr.shape, "| X_te:", X_te.shape)

X_tr: (143613, 200) | y_tr: (143613, 6) | X_te: (153164, 200)


# GloVe

In [19]:
# GloVe — предобученные векторы слов; стартуем эмбеддинги с них, а не учим с нуля
def load_glove(path):
    emb = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            emb[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
    return emb

glove = load_glove(GLOVE_PATH)

In [ ]:
# строка i матрицы = вектор GloVe для itos[i]; чего нет в GloVe — мелкий шум, pad — нули
rng = np.random.default_rng(SEED)
embedding_matrix = rng.normal(0, 0.1, (VOCAB_SIZE, EMBED_DIM)).astype(np.float32)
embedding_matrix[PAD_IDX] = 0.0

found = 0
for word, idx in stoi.items():
    vec = glove.get(word)
    if vec is not None:
        embedding_matrix[idx] = vec
        found += 1
print(f"покрытие GloVe: {found}/{VOCAB_SIZE} ({found/VOCAB_SIZE:.1%})")

покрытие GloVe: 41922/50000 (83.8%)


# Dataset / DataLoader

In [ ]:
# Dataset — индексируемый контейнер: __len__ сколько примеров, __getitem__ один (x, y)
class ToxicDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.as_tensor(X, dtype=torch.long)   # id -> long
        self.y = None if y is None else torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        if self.y is None:
            return self.X[i]            
        return self.X[i], self.y[i]

# DataLoader выдаёт батчами; shuffle на train — чтобы модель не учила порядок
train_loader = DataLoader(ToxicDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ToxicDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(ToxicDataset(X_te), batch_size=BATCH_SIZE, shuffle=False)

xb, yb = next(iter(train_loader))
print("батч:", xb.shape, yb.shape)

батч: torch.Size([64, 200]) torch.Size([64, 6])


# Модель

In [ ]:
# Embedding -> BiLSTM -> Dropout -> Linear -> Sigmoid
class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=HIDDEN_DIM,
                 num_labels=len(LABELS), dropout=DROPOUT, freeze=True):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.as_tensor(embedding_matrix), freeze=freeze, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)         
        self.fc = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, x):
        emb = self.embedding(x)                    # (batch, seq, embed)
        out, (h_n, c_n) = self.lstm(emb)           # h_n: (2, batch, hidden)
        # h_n[0] — прямое направление, h_n[1] — обратное; конкат = вектор всего коммента
        sent = torch.cat([h_n[0], h_n[1]], dim=1)  # (batch, 2*hidden)
        sent = self.dropout(sent)
        logits = self.fc(sent)                     # (batch, 6) сырые скоры
        return torch.sigmoid(logits)

model = BiLSTMClassifier(embedding_matrix).to(device)   # кладём параметры на device
print(model)
print("обучаемых параметров:", sum(p.numel() for p in model.parameters() if p.requires_grad))

BiLSTMClassifier(
  (embedding): Embedding(50000, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=6, bias=True)
)
обучаемых параметров: 237062


In [12]:
# BCELoss: бинарная кросс-энтропия по каждой из 6 меток отдельно, ждёт вероятности [0,1]
criterion = nn.BCELoss()
# Adam: адаптивный lr; filter — обновляем только незамороженные параметры (не эмбеддинги)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

# Обучение

In [13]:
# eval: model.eval() выключает dropout, no_grad() не строит граф (экономия памяти, без backward)
@torch.no_grad()
def evaluate(loader, has_labels=True):
    model.eval()
    probs_all, targets_all, losses = [], [], []
    for batch in loader:
        if has_labels:
            x, y = batch
            x, y = x.to(device), y.to(device)
            probs = model(x)
            losses.append(criterion(probs, y).item())
            targets_all.append(y.cpu().numpy())
        else:
            probs = model(batch.to(device))
        probs_all.append(probs.cpu().numpy())   # обратно на cpu для numpy/sklearn
    probs = np.concatenate(probs_all)
    targets = np.concatenate(targets_all) if has_labels else None
    loss = float(np.mean(losses)) if has_labels else None
    return loss, probs, targets

# ROC-AUC, усреднённый по 6 столбцам (метрика Kaggle)
def mean_auc(targets, probs):
    aucs = []
    for j in range(len(LABELS)):
        if len(np.unique(targets[:, j])) < 2:   # столбец из одних нулей -> AUC не определён
            aucs.append(float("nan"))
        else:
            aucs.append(roc_auc_score(targets[:, j], probs[:, j]))
    return float(np.nanmean(aucs)), aucs

In [14]:
best_auc = -1.0
ckpt_path = "../checkpoints/bilstm_best.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()                       # режим обучения: dropout включён
    running, t0 = 0.0, time.time()
    for x, y in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()           # обнуляем градиенты — PyTorch их накапливает
        probs = model(x)                # прямой проход
        loss = criterion(probs, y)      # насколько ошиблись
        loss.backward()                 # обратное распространение: считает .grad всех весов
        optimizer.step()                # шаг оптимизатора: обновляет веса по градиентам
        running += loss.item() * x.size(0)
    train_loss = running / len(tr)

    # валидация после каждой эпохи
    val_loss, val_probs, val_targets = evaluate(val_loader)
    val_auc, _ = mean_auc(val_targets, val_probs)
    print(f"epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
          f"val_auc={val_auc:.4f} ({time.time()-t0:.0f}s)")

    # сохраняем лучший чекпойнт — state_dict (только веса), рекомендованный способ
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({"model_state": model.state_dict(), "stoi": stoi, "itos": itos,
                    "val_auc": best_auc}, ckpt_path)
        print(f"  сохранил лучший -> {ckpt_path} (auc {best_auc:.4f})")

print("лучший val AUC:", round(best_auc, 4))

epoch 1/4:   0%|          | 0/2244 [00:00<?, ?it/s]

epoch 1: train_loss=0.0769 val_loss=0.0554 val_auc=0.9684 (77s)
  сохранил лучший -> ../checkpoints/bilstm_best.pt (auc 0.9684)


epoch 2/4:   0%|          | 0/2244 [00:00<?, ?it/s]

epoch 2: train_loss=0.0558 val_loss=0.0509 val_auc=0.9746 (76s)
  сохранил лучший -> ../checkpoints/bilstm_best.pt (auc 0.9746)


epoch 3/4:   0%|          | 0/2244 [00:00<?, ?it/s]

epoch 3: train_loss=0.0512 val_loss=0.0471 val_auc=0.9797 (74s)
  сохранил лучший -> ../checkpoints/bilstm_best.pt (auc 0.9797)


epoch 4/4:   0%|          | 0/2244 [00:00<?, ?it/s]

epoch 4: train_loss=0.0478 val_loss=0.0464 val_auc=0.9818 (74s)
  сохранил лучший -> ../checkpoints/bilstm_best.pt (auc 0.9818)
лучший val AUC: 0.9818


# Оценка

In [15]:
# грузим лучший чекпойнт и смотрим AUC по каждой метке
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])

_, val_probs, val_targets = evaluate(val_loader)
val_auc, per_label = mean_auc(val_targets, val_probs)
for name, a in zip(LABELS, per_label):
    print(f"{name:14s}: {a:.4f}")
print(f"{'MEAN':14s}: {val_auc:.4f}")

toxic         : 0.9775
severe_toxic  : 0.9856
obscene       : 0.9881
threat        : 0.9783
insult        : 0.9846
identity_hate : 0.9770
MEAN          : 0.9818


# Сабмит

In [16]:
# прогоняем тест, берём 6 вероятностей на коммент, пишем в формате Kaggle
_, test_probs, _ = evaluate(test_loader, has_labels=False)
submission = pd.DataFrame(test_probs, columns=LABELS)
submission.insert(0, "id", test["id"].values)
submission.to_csv("../submissions/submission_bilstm.csv", index=False)
print("submission:", submission.shape)
submission.head()

submission: (153164, 7)


,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.995141,0.243665,0.962152,0.105214,0.908952,0.451512
1,0000247867823ef7,0.001338,0.000017,0.000396,0.000026,0.000326,0.000071
2,00013b17ad220c46,0.032146,0.000475,0.012492,0.000550,0.009231,0.002312
3,00017563c3f7919a,0.001719,0.000023,0.000184,0.000117,0.000260,0.000076
4,00017695ad8997eb,0.005254,0.000274,0.001120,0.000681,0.001633,0.000409


GloVe замороженный — можно потом разморозить (`requires_grad=True`) и дообучить. BCEWithLogitsLoss стабильнее и умеет `pos_weight` против дисбаланса классов.